# KG1 v143 — Nemotron-3-Nano-30B SFT (Colab Pro+ A100/H100)

## Versao revisada LINHA POR LINHA - testada com 8 falhas reais

**Mudancas vs v142**:
- Cada cell totalmente self-contained (imports inline)
- `dtype=` em vez de `torch_dtype=` (deprecated)
- Token reload + validate em todos cells que precisam
- Validation gates apos cada step critico
- Cell 0 PRE-FLIGHT antes de install
- Robust GDrive backup (Cell 10 sempre funciona)

## Datasets (~11,402 CoTs total, public sem auth):
- **Tong corpus** → 6,014 (`felipesp1983/nemotron-cot-tong-dgxchen`)
- **Cryptarithm 813** → 813 (`felipesp1983/nemotron-cryptarithm-cot-813`)
- **EqGuess 150** → 150 (gist DeepSeek-R1 distill)
- **Synth Z3 4425** → 4,425 (`felipesp1983/nemotron-cryptarithm-synth-4425`)

## Hyperparams (consenso 28 APIs triple check)
- LoRA r=32 alpha=32 all-linear+lm_head
- lr=5e-5, epochs=2, cosine, warmup=0.03, weight_decay=0.01
- max_length=8192, batch effective 32
- Curriculum: tong → eq_guess → cryptarithm_813 → synth_4425

## Setup obrigatorio

1. **Runtime → Change runtime type → A100** (HighRAM se disponivel) ou **H100**
2. **Colab Secrets** (icone chave esquerdo):
   - `HF_KEY` (token Write+Jobs+Read gated repos)
   - `KAGGLE_USERNAME`, `KAGGLE_KEY`
3. **Aceitar termos** em https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
4. **Runtime → Run all**

Tempo: 3-4h em H100, 4-6h em A100

## Versoes verificadas (Apr 2026)
- mamba-ssm: 2.3.1 (suporte torch 2.8/2.9/2.10/2.11)
- causal-conv1d: 1.6.1 (release tag v1.6.1.post4)
- transformers: 4.48+, peft: 0.14+, trl: 0.14+


In [ ]:
# CELL 0: PRE-FLIGHT - sanity check antes de instalar nada
import os, sys, time
print('=' * 60)
print('CELL 0: PRE-FLIGHT CHECK')
print('=' * 60)

# 1. Disable HF telemetry (evita warnings inuteis)
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'  # mantem progress bars
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

# 2. Detect Colab + secrets
try:
    from google.colab import userdata
    IS_COLAB = True
    print('[OK] Google Colab detected')
except ImportError:
    IS_COLAB = False
    print('[WARN] Not in Colab')

# 3. HF token (HF_KEY preferred)
hf_token = None
if IS_COLAB:
    for key_name in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(key_name)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                os.environ['HF_KEY'] = v
                print(f'[OK] HF token loaded via {key_name} (len={len(v)})')
                break
        except Exception:
            continue

    if not hf_token:
        print('[ERR] HF_KEY/HF_TOKEN nao configurado nos Colab Secrets')
        print('      1. Click no icone chave (lado esquerdo)')
        print('      2. + Add new secret')
        print('      3. Name: HF_KEY')
        print('      4. Value: seu token (gere em https://huggingface.co/settings/tokens)')
        print('      5. Permissoes: Read access gated repos + Write repos + Inference')
        raise RuntimeError('HF_KEY missing')

    # Kaggle
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print(f'[OK] Kaggle creds: {os.environ["KAGGLE_USERNAME"]}')
    except Exception:
        print('[INFO] Kaggle creds not configured (optional pra training)')

# 4. GPU check
import torch
print()
print(f'[INFO] PyTorch: {torch.__version__}')
print(f'[INFO] CUDA: {torch.version.cuda}')
print(f'[INFO] cxx11_abi: {torch.compiled_with_cxx11_abi()}')

if not torch.cuda.is_available():
    print('[ERR] GPU nao disponivel!')
    print('      Runtime > Change runtime type > GPU > A100 ou H100')
    raise RuntimeError('GPU required')

gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'[OK] GPU: {gpu_name}')
print(f'[OK] VRAM: {vram:.1f} GB')

if vram < 35:
    print('[WARN] VRAM < 35 GB pode dar OOM no model 30B')
    print('       Reconfigure para A100 (40GB) ou H100 (80GB)')

import psutil
print(f'[OK] RAM: {psutil.virtual_memory().total/1e9:.1f} GB')

# 5. Disk space check
disk = os.statvfs('/content')
free_gb = (disk.f_frsize * disk.f_bavail) / 1e9
print(f'[OK] Free disk in /content: {free_gb:.1f} GB')
if free_gb < 60:
    print('[WARN] Low disk - model 30B precisa ~60GB livres')

# 6. Validate HF token
print()
print('Validating HF token...')
try:
    from huggingface_hub import HfApi
    whoami = HfApi(token=hf_token).whoami()
    print(f'[OK] HF token VALID, user: {whoami["name"]}')
    # Check fine-grained permissions
    auth = whoami.get('auth', {}).get('accessToken', {})
    fg = auth.get('fineGrained', {})
    if fg.get('canReadGatedRepos'):
        print('[OK] canReadGatedRepos: True (Nemotron acessivel)')
    else:
        print('[WARN] canReadGatedRepos: False - vai falhar no Cell 6!')
        print('       Edite o token em https://huggingface.co/settings/tokens')
        print('       e marque "Read access to public gated repos"')
except Exception as e:
    print(f'[ERR] HF token invalido: {e}')
    print('      Gere novo em https://huggingface.co/settings/tokens')
    print('      Atualize Colab Secret HF_KEY')
    raise

print()
print('=' * 60)
print('PRE-FLIGHT OK - pode prosseguir para Cell 1')
print('=' * 60)


In [ ]:
# CELL 1: Install Python dependencies (Colab tem CUDA + PyTorch preinstalled)
import sys, subprocess

print('=' * 60)
print('CELL 1: Install dependencies')
print('=' * 60)

# Install via pip (silent)
deps = [
    'transformers>=4.48.0',
    'peft>=0.14.0',
    'trl>=0.14.0',
    'datasets>=3.0.0',
    'accelerate>=1.3.0',
    'bitsandbytes>=0.45.0',
    'huggingface_hub>=0.27.0',
    'einops',
    'packaging',
    'ninja',
    'triton',
]

import time
t0 = time.time()
for pkg in deps:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                      capture_output=True, text=True, timeout=300)
    status = 'OK' if r.returncode == 0 else 'FAIL'
    print(f'  [{status}] {pkg}')
    if r.returncode != 0:
        print(f'    err: {r.stderr[-200:]}')

# Verify
print()
print('Verifying versions...')
import importlib
for mod_name in ['transformers', 'peft', 'trl', 'datasets', 'accelerate', 'bitsandbytes', 'huggingface_hub']:
    try:
        mod = importlib.import_module(mod_name)
        v = getattr(mod, '__version__', 'imported')
        print(f'  [OK] {mod_name}: {v}')
    except ImportError as e:
        print(f'  [FAIL] {mod_name}: {e}')

print(f'\nTotal time: {time.time()-t0:.1f}s')


In [ ]:
# CELL 2: Install mamba-ssm 2.3.1 + causal-conv1d 1.6.1 (Colab Python 3.12 + Torch 2.10)
import sys, subprocess
import torch

print('=' * 60)
print('CELL 2: Install mamba-ssm + causal-conv1d (CUDA kernels)')
print('=' * 60)

# Detect Python + Torch + ABI
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'  # cp312
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])  # 2.10
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'
cuda_str = 'cu12'  # works for any cuda 12.x
print(f'Detected: {py_ver} | torch{torch_short} | {cuda_str} | abi{abi_str}')

MAMBA_VER = '2.3.1'
CC_TAG = '1.6.1.post4'
CC_ASSET_VER = '1.6.1'

def install_combo(cu, t, abi):
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v{CC_TAG}/causal_conv1d-{CC_ASSET_VER}+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v{MAMBA_VER}/mamba_ssm-{MAMBA_VER}+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    for url in [cc_url, mb_url]:
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', url],
            capture_output=True, text=True, timeout=600
        )
        if r.returncode != 0:
            return False, r.stderr[-200:]
    return True, ''

# Try detected combo first, then fallbacks (same major torch)
attempts = [(cuda_str, torch_short, abi_str)]
opposite_abi = 'FALSE' if abi_str == 'TRUE' else 'TRUE'
attempts.append((cuda_str, torch_short, opposite_abi))
for t in ['2.10', '2.9', '2.8']:
    for abi in ['TRUE', 'FALSE']:
        combo = (cuda_str, t, abi)
        if combo not in attempts:
            attempts.append(combo)

success = False
for cu, t, abi in attempts:
    print(f'  Trying: {cu} torch{t} abi{abi}...')
    ok, err = install_combo(cu, t, abi)
    if ok:
        print(f'  [OK] Installed: mamba_ssm={MAMBA_VER} causal_conv1d={CC_ASSET_VER} ({cu} torch{t} abi{abi})')
        success = True
        break
    else:
        print(f'  [FAIL] {err[:120].replace(chr(10), " ")}')

if not success:
    print()
    print('All wheel combos failed. Last fallback: build from source...')
    import os
    os.environ['CAUSAL_CONV1D_FORCE_BUILD'] = 'TRUE'
    os.environ['MAMBA_FORCE_BUILD'] = 'TRUE'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                    '--force-reinstall', f'causal-conv1d=={CC_ASSET_VER}'], timeout=1200)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                    '--force-reinstall', f'mamba-ssm=={MAMBA_VER}'], timeout=1500)

# Force reload of cached modules
for m in ['mamba_ssm', 'causal_conv1d', 'selective_scan_cuda', 'causal_conv1d_cuda']:
    if m in sys.modules:
        del sys.modules[m]

# Verify CUDA kernels available
print()
print('=== Verification ===')
all_ok = True
for m in ['mamba_ssm', 'causal_conv1d', 'selective_scan_cuda', 'causal_conv1d_cuda']:
    try:
        mod = __import__(m)
        v = getattr(mod, '__version__', 'imported')
        print(f'  [OK] {m}: {v}')
    except ImportError as e:
        print(f'  [FAIL] {m}: {e}')
        all_ok = False

if not all_ok:
    raise RuntimeError('mamba-ssm/causal-conv1d incomplete - cannot proceed to model load')

print()
print('SUCCESS: All mamba-ssm CUDA kernels available!')


In [ ]:
# CELL 3: Download all 4 datasets via direct HTTP (sem precisar de token)
import os, urllib.request, time

print('=' * 60)
print('CELL 3: Download datasets (4 fontes, ~52 MB total)')
print('=' * 60)

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

URLS = {
    'tong.csv': 'https://huggingface.co/datasets/felipesp1983/nemotron-cot-tong-dgxchen/resolve/main/less_cot.csv',
    'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
    'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
    'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
}

t0 = time.time()
for name, url in URLS.items():
    out = os.path.join(DATA_DIR, name)
    print(f'  Downloading {name}...')
    try:
        urllib.request.urlretrieve(url, out)
        sz = os.path.getsize(out)
        if sz < 1000:
            raise RuntimeError(f'File too small: {sz} bytes (likely error)')
        if sz > 1e6:
            print(f'    [OK] {sz/1e6:.2f} MB')
        else:
            print(f'    [OK] {sz/1e3:.1f} KB')
    except Exception as e:
        print(f'    [FAIL] {e}')
        raise

# Set globals
DATA_PATHS = {
    'tong': os.path.join(DATA_DIR, 'tong.csv'),
    'crypt_813': os.path.join(DATA_DIR, 'cryptarithm_813.jsonl'),
    'eq_guess_150': os.path.join(DATA_DIR, 'eq_guess_150.jsonl'),
    'synth_4425': os.path.join(DATA_DIR, 'synth_4425.jsonl'),
}

print(f'\n[OK] All 4 datasets downloaded in {time.time()-t0:.1f}s')
print('Continue to Cell 4.')


In [ ]:
# CELL 4: Format + Merge + Curriculum (easy -> hard)
import os, csv, json
from datasets import Dataset

print('=' * 60)
print('CELL 4: Format + Merge + Curriculum')
print('=' * 60)

# Verify Cell 3 ran
DATA_DIR = '/content/data'
DATA_PATHS = {
    'tong': os.path.join(DATA_DIR, 'tong.csv'),
    'crypt_813': os.path.join(DATA_DIR, 'cryptarithm_813.jsonl'),
    'eq_guess_150': os.path.join(DATA_DIR, 'eq_guess_150.jsonl'),
    'synth_4425': os.path.join(DATA_DIR, 'synth_4425.jsonl'),
}
for name, p in DATA_PATHS.items():
    if not os.path.exists(p):
        raise FileNotFoundError(f'Missing {name}: {p} - run Cell 3 first')

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

all_data = []

# 1. Tong corpus
n_tong = 0
with open(DATA_PATHS['tong'], encoding='utf-8') as f:
    rd = csv.DictReader(f)
    for row in rd:
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({'prompt': prompt + PROMPT_SUFFIX, 'completion': cot, 'source': 'tong'})
            n_tong += 1

# 2. Cryptarithm 813
n_crypt = 0
with open(DATA_PATHS['crypt_813'], encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if row.get('is_valid') and row.get('cot'):
            all_data.append({'prompt': row['prompt'].strip() + PROMPT_SUFFIX,
                             'completion': row['cot'].strip(), 'source': 'cryptarithm_813'})
            n_crypt += 1

# 3. EqGuess 150
n_eq = 0
with open(DATA_PATHS['eq_guess_150'], encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({'prompt': prompt + PROMPT_SUFFIX, 'completion': cot, 'source': 'eq_guess_150'})
            n_eq += 1

# 4. Synth Z3 4425
n_synth = 0
with open(DATA_PATHS['synth_4425'], encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({'prompt': prompt + PROMPT_SUFFIX, 'completion': cot, 'source': 'synth_4425'})
            n_synth += 1

print(f'  Tong:               {n_tong:>6}')
print(f'  Cryptarithm 813:    {n_crypt:>6}')
print(f'  EqGuess 150:        {n_eq:>6}')
print(f'  Synth Z3 4425:      {n_synth:>6}')
print(f'  TOTAL:              {len(all_data):>6}')

# Sanity check
expected_total = 11402
if abs(len(all_data) - expected_total) > 100:
    print(f'  [WARN] Total {len(all_data)} != expected ~{expected_total}')

# Curriculum
SOURCE_ORDER = {'tong': 0, 'eq_guess_150': 1, 'cryptarithm_813': 2, 'synth_4425': 3}
all_data.sort(key=lambda x: SOURCE_ORDER.get(x['source'], 99))

ds = Dataset.from_list(all_data)
print(f'\n[OK] Dataset ready: {len(ds)} samples')
print('Curriculum: tong -> eq_guess -> cryptarithm_813 -> synth_4425')


In [ ]:
# CELL 5: Load Nemotron-3-Nano-30B + LoRA (auto NF4 if VRAM < 70GB)
import os, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print('=' * 60)
print('CELL 5: Load Nemotron-30B + LoRA')
print('=' * 60)

# === Reload + validate HF token (caso tenha sido renovado) ===
try:
    from google.colab import userdata
    for key_name in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(key_name)
            if v:
                os.environ['HF_TOKEN'] = v
                break
        except Exception:
            continue
except ImportError:
    pass

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN missing - configure HF_KEY in Colab Secrets')

# Validate
from huggingface_hub import HfApi
try:
    api = HfApi(token=hf_token)
    whoami = api.whoami()
    print(f'  [OK] HF token, user: {whoami["name"]}')
    fg = whoami.get('auth', {}).get('accessToken', {}).get('fineGrained', {})
    if not fg.get('canReadGatedRepos'):
        print('  [WARN] canReadGatedRepos=False - Nemotron download pode falhar')
except Exception as e:
    print(f'  [ERR] HF token invalid: {e}')
    print('  Gere novo em https://huggingface.co/settings/tokens (com Read gated)')
    raise

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

# Auto-detect VRAM -> NF4 conditional
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
USE_NF4 = vram_gb < 70  # H100/A100 80GB = no NF4
print(f'  VRAM: {vram_gb:.1f} GB -> USE_NF4: {USE_NF4}')

# Load tokenizer
print('\nLoading tokenizer...')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'  [OK] tokenizer in {time.time()-t0:.1f}s')

# Load model (use 'dtype' nao 'torch_dtype' - deprecated)
print(f'\nLoading model 30B (NF4={USE_NF4}) - download 60GB + load 5-10 min...')
t0 = time.time()
if USE_NF4:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={'': 0},
        trust_remote_code=True,
        token=hf_token,
        attn_implementation='eager',
        dtype=torch.bfloat16,                 # NEW API (era torch_dtype)
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.bfloat16,                 # NEW API (era torch_dtype)
        device_map='auto',
        trust_remote_code=True,
        token=hf_token,
        attn_implementation='eager',
    )
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable()

print(f'  [OK] Model loaded in {time.time()-t0:.1f}s')
print(f'  VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB / {vram_gb:.1f} GB')

# Apply LoRA
TARGET_REGEX = r'.*\.(in_proj|out_proj|q_proj|k_proj|v_proj|o_proj|up_proj|down_proj|gate_proj|lm_head)$'
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=32, target_modules=TARGET_REGEX,
    lora_dropout=0.0, bias='none', task_type='CAUSAL_LM',
))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'  [OK] LoRA r=32 alpha=32: {trainable/1e6:.1f}M trainable / {total/1e9:.2f}B total ({100*trainable/total:.3f}%)')
print(f'  VRAM after LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB / {vram_gb:.1f} GB')
print('\nSUCCESS - continue to Cell 6')


In [ ]:
# CELL 6: Tokenize com chat template + prompt masking
import time

print('=' * 60)
print('CELL 6: Tokenize (chat template + prompt masking)')
print('=' * 60)

# Requires: tokenizer (Cell 5), ds (Cell 4)
# Sanity check
if 'tokenizer' not in dir():
    raise NameError('tokenizer not defined - run Cell 5 first')
if 'ds' not in dir():
    raise NameError('ds not defined - run Cell 4 first')

MAX_LENGTH = 8192

def fmt(ex):
    messages = [
        {'role': 'user', 'content': ex['prompt']},
        {'role': 'assistant', 'content': ex['completion']},
    ]
    prompt_ids = tokenizer.apply_chat_template(
        [messages[0]], tokenize=True,
        add_generation_prompt=True, enable_thinking=True,
    )
    full_ids = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=False, enable_thinking=True,
    )
    if len(full_ids) > MAX_LENGTH:
        full_ids = full_ids[:MAX_LENGTH]

    labels = list(full_ids)
    n_prompt = min(len(prompt_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100

    return {
        'input_ids': full_ids,
        'attention_mask': [1] * len(full_ids),
        'labels': labels,
    }

print(f'Tokenizing {len(ds)} samples (max_length={MAX_LENGTH})...')
t0 = time.time()
tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
print(f'\n[OK] Tokenized {len(tokenized)} samples in {time.time()-t0:.1f}s')

# Sanity check first sample
sample = tokenized[0]
n_total = len(sample['input_ids'])
n_masked = sum(1 for x in sample['labels'] if x == -100)
n_train = sum(1 for x in sample['labels'] if x != -100)
print(f'\nFirst sample:')
print(f'  total tokens:       {n_total}')
print(f'  prompt (masked):    {n_masked}  (loss=0 nesses)')
print(f'  completion (train): {n_train}   (loss calculado nesses)')


In [ ]:
# CELL 7: TRAIN (lr=5e-5, epochs=2, cosine - consenso 28 APIs)
import os, time
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, TrainerCallback

print('=' * 60)
print('CELL 7: TRAIN - 2 epochs, lr 5e-5, cosine')
print('=' * 60)

# Sanity checks
if 'model' not in dir():
    raise NameError('model not defined - run Cell 5 first')
if 'tokenized' not in dir():
    raise NameError('tokenized not defined - run Cell 6 first')

OUTPUT_DIR = '/content/output_v143'
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    adam_beta1=0.9,
    adam_beta2=0.95,
    adam_epsilon=1e-8,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    bf16=True,
    optim='adamw_torch',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    remove_unused_columns=False,
    report_to='none',
    dataloader_num_workers=2,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors='pt',
)


# Loss explosion detection (early stopping if loss goes haywire)
class LossExplosionCallback(TrainerCallback):
    def __init__(self, threshold=15.0):
        self.threshold = threshold
        self.first_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or 'loss' not in logs:
            return
        loss = logs['loss']
        if self.first_loss is None:
            self.first_loss = loss
        if loss > self.threshold or (loss != loss):  # NaN check
            print(f'\n[WARN] Loss explosion at step {state.global_step}: {loss}')
            print('       Stopping training - check hyperparams')
            control.should_training_stop = True


trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized,
    data_collator=data_collator,
    callbacks=[LossExplosionCallback(threshold=15.0)],
)

n_steps = (len(tokenized) // 32) * 2
print(f'Training {len(tokenized)} samples for 2 epochs (effective batch 32)...')
print(f'Expected: ~{n_steps} optimizer steps')
print(f'Time estimate: ~2-3h H100, ~4-6h A100')
print()

t0 = time.time()
trainer.train()
train_time_min = (time.time() - t0) / 60
print()
print(f'[OK] Training complete in {train_time_min:.1f} min')


In [ ]:
# CELL 8: Save adapter + Upload to HuggingFace
import os
print('=' * 60)
print('CELL 8: Save + Upload to HF')
print('=' * 60)

# Sanity check
if 'trainer' not in dir():
    raise NameError('trainer not defined - run Cell 7 first')

OUTPUT_DIR = '/content/output_v143'
ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'

# Always save locally first
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'[OK] Adapter saved at {ADAPTER_DIR}')
print('\nFiles:')
for f in sorted(os.listdir(ADAPTER_DIR)):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f'  {f}: {sz/1e6:.2f} MB')

# === RELOAD HF token (case it expired during 3h training) ===
try:
    from google.colab import userdata
    for key_name in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(key_name)
            if v:
                os.environ['HF_TOKEN'] = v
                break
        except Exception:
            continue
except ImportError:
    pass

hf_token = os.environ.get('HF_TOKEN')

DEST_REPO = 'felipesp1983/kg1-v143-final'

if not hf_token:
    print('\n[ERR] HF_TOKEN missing - upload skipped')
    print('     Cell 9 (GDrive backup) will save the adapter to your Drive')
else:
    # Validate token before uploading
    try:
        from huggingface_hub import HfApi, create_repo
        api = HfApi(token=hf_token)
        whoami = api.whoami()
        print(f'\nHF user: {whoami["name"]}')

        create_repo(DEST_REPO, private=False, exist_ok=True, token=hf_token)
        api.upload_folder(
            folder_path=ADAPTER_DIR,
            repo_id=DEST_REPO,
            repo_type='model',
            commit_message='v143 SFT Tong+813+150+4425synth lr5e-5 ep2 cosine',
        )
        print(f'\n[OK] Uploaded to: https://huggingface.co/{DEST_REPO}')
    except Exception as e:
        print(f'\n[ERR] Upload failed: {e}')
        print('\nGere novo token HF (https://huggingface.co/settings/tokens)')
        print('Atualize Colab Secret HF_KEY')
        print('Re-rode SOMENTE este Cell 8 (Cell 9 GDrive sempre funciona)')


In [ ]:
# CELL 9: GDrive backup (SEMPRE funciona, mesmo se Cell 8 falhar)
import os, shutil, subprocess

print('=' * 60)
print('CELL 9: GDrive backup')
print('=' * 60)

# Find adapter dir (may be from Cell 7 trainer or Cell 8)
ADAPTER_DIR = '/content/output_v143/final_adapter'
if not os.path.exists(ADAPTER_DIR):
    # Try alternative locations
    for cand in ['/content/output_v142/final_adapter', '/content/output/final_adapter']:
        if os.path.exists(cand):
            ADAPTER_DIR = cand
            print(f'[INFO] Using fallback: {ADAPTER_DIR}')
            break
    else:
        raise FileNotFoundError('Adapter dir not found - Cells 7+8 must run first')

# Mount drive
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'[WARN] Drive mount: {e}')

GDRIVE_BACKUP = '/content/drive/My Drive/KG1_v143_adapter'

# Remove old backup if exists
if os.path.exists(GDRIVE_BACKUP):
    print(f'  Removing old backup at {GDRIVE_BACKUP}...')
    shutil.rmtree(GDRIVE_BACKUP)

# Copy
shutil.copytree(ADAPTER_DIR, GDRIVE_BACKUP)
print(f'[OK] Backup at: {GDRIVE_BACKUP}')

# Show size
sz = subprocess.run(['du', '-sh', GDRIVE_BACKUP], capture_output=True, text=True).stdout.strip()
print(f'  Size: {sz}')

# List files
print('\nFiles:')
for f in sorted(os.listdir(GDRIVE_BACKUP)):
    p = os.path.join(GDRIVE_BACKUP, f)
    if os.path.isfile(p):
        sz_f = os.path.getsize(p)
        print(f'  {f}: {sz_f/1e6:.2f} MB')

print()
print('IMPORTANT: keep this Drive backup as source-of-truth.')
print('Se HF upload falhar voce ainda tem o adapter salvo aqui.')


## Proximos passos apos v143 treinar

### 1. Quick eval no Colab (opcional, ~5 min)

Adicione em uma nova cell:
```python
# Test adapter generates correct \boxed{} format
test_prompt = "In Alice's Wonderland, a secret set of transformation rules is applied to equations."
inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)
output = model.generate(**inputs, max_new_tokens=512, temperature=0)
print(tokenizer.decode(output[0], skip_special_tokens=False))
```

### 2. Submit Kaggle (terminal local depois)

```bash
python scripts/submit_kaggle.py \\
    --hf-repo felipesp1983/kg1-v143-final \\
    --reference-adapter-dir runs/attached_085_audit_20260416 \\
    --test-csv data/kaggle/unzipped/test.csv \\
    --message "v143 11400CoTs Tong+synth4425 lr5e-5 ep2 cosine" \\
    --max-daily-submits 5
```

### 3. Score esperado (consenso 28 APIs)
- **Mediana**: 0.92
- Range: [0.88 - 0.95]
- P(>= 0.87): 75-85%

### 4. Se v143 < 0.87
Proximo: **v144 GRPO** sobre v143 SFT
- Adaptar `rewards.py` do andy279 (open-r1 fork)
- Verifiable reward via Z3 (cryptarithm) + sympy (eq_numeric)
- +5-10% adicional esperado

## Troubleshooting

| Erro | Solucao |
|---|---|
| Cell 0 token invalid | Gere novo em huggingface.co/settings/tokens com `Read gated repos` |
| Cell 2 mamba-ssm fail | Auto-detecta torch/python/abi e tenta 8 combos. Se falha, build from source |
| Cell 5 NameError | Cell 5 totalmente self-contained - so precisa Cell 0-2 antes |
| Cell 5 OOM | Verifica se VRAM esta mostrando GB esperado, usa NF4 em < 70GB automaticamente |
| Cell 7 loss explode | EarlyStoppingCallback para automaticamente em loss > 15 ou NaN |
| Cell 8 token expired | Gere novo, atualize Secret, re-rode Cell 8 (Cell 9 GDrive sempre funciona) |
| Cell 9 Drive issue | Tenta `drive.mount('/content/drive', force_remount=True)` |

## Cell ordering (rode em ordem)

```
Cell 0: PRE-FLIGHT (3s)
Cell 1: pip deps (2 min)
Cell 2: mamba-ssm (1-3 min)
Cell 3: download datasets (30s)
Cell 4: format + curriculum (5s)
Cell 5: load model + LoRA (5-10 min)
Cell 6: tokenize (1 min)
Cell 7: TRAIN (2-6h)
Cell 8: HF upload (3-5 min)
Cell 9: GDrive backup (1-2 min)
```
